# Lift and Learn (Least Squares) example for Fitzhugh-Nagumo Equation

## Fitzhugh-Nagumo Equation

The Fitzhugh-Nagumo equation is a simplified mathematical model used to describe the behavior of excitable systems, particularly the dynamics of nerve cells or neurons. It was developed independently by Richard Fitzhugh and John Nagumo in the 1960s as a modification of the Hodgkin-Huxley model, aiming to capture the essential characteristics of action potential propagation.

The equation takes the form of a system of ordinary differential equations and consists of two main variables: a fast-recovery variable (usually denoted as v) and a slow-activating variable (usually denoted as w). The Fitzhugh-Nagumo equation is given by:
$$
\begin{align*}
    \frac{dv}{dt} &= v - \frac{v^3}{3} - w + I \\
    \frac{dw}{dt} &= \epsilon (v + a - bw)
\end{align*}
$$
where:
- v represents the membrane potential of the neuron.
- w represents a recovery variable related to the inactivation of ion channels.
- I is an external input current that stimulates the neuron.
- ε is a positive parameter that controls the timescale separation between v and w dynamics.
- a and b are constants that affect the equilibrium points and behavior of the system.

The Fitzhugh-Nagumo model simplifies the complex behavior of neurons, focusing on the interplay between the fast-responding action potential and the slower recovery process. This simplification makes it easier to analyze and understand the fundamental mechanisms underlying excitable systems. The Fitzhugh-Nagumo equation has been widely used in the study of neuronal dynamics, cardiac rhythm modeling, and other excitable phenomena, providing insights into the generation and propagation of electrical signals in various biological and physical contexts.

## The FitzHugh-Nagumo PDE used here

In this example we use the spatially extended (PDE) version of the model from the Lift & Learn paper (Qian et al., 2020), implemented as `FitzHughNagumoModel` in [PolynomialModelReductionDataset.jl](https://github.com/smallpondtom/PolynomialModelReductionDataset.jl):
$$
\begin{align*}
    \gamma \frac{\partial u}{\partial t} &= \gamma^2 \frac{\partial^2 u}{\partial x^2} + u(u - 0.1)(1 - u) - v + g, \\
    \frac{\partial v}{\partial t} &= R\,u - c\,v + g,
\end{align*}
$$
on $x \in [0, 1]$, $t \in [0, 4]$ with $\gamma = 0.015$, $R = 0.5$, $c = 2$, $g = 0.05$, zero initial conditions and the Neumann boundary conditions $u_x(0,t) = -i_0(t)$, $u_x(1,t) = 0$. The input current is $i_0(t) = \alpha t^3 e^{-\beta t}$ where the parameters $\alpha$ and $\beta$ are varied.

The cubic nonlinearity makes the model non-polynomial of degree two. **Lift & Learn** introduces the auxiliary variable $w = u^2$, which turns the system into a *quadratic-bilinear* system in the lifted state $(u, v, w)$:
$$
\dot{\mathbf{w}} = \mathbf{A}\mathbf{w} + \mathbf{B}u_{\text{in}} + \mathbf{A}_2(\mathbf{w}\otimes\mathbf{w}) + \mathbf{N}\mathbf{w}\,u_{\text{in}} + \mathbf{K} .
$$
We then learn reduced operators of this lifted structure from (lifted) snapshot data, and compare them against the intrusive POD-Galerkin projection of the lifted full-order operators.

## Setup

In [ ]:
using BlockDiagonals
using LinearAlgebra
using NaNStatistics
using Plots
using ProgressMeter
using Random
using SparseArrays
using Statistics
using UniqueKronecker
using PolynomialModelReductionDataset: FitzHughNagumoModel
using LiftAndLearn
const LnL = LiftAndLearn

Random.seed!(1234)  # reproducible random test parameters
;

## Generate the Models

In [ ]:
start = time()
Ω = (0.0, 1.0); dt = 1e-4; Nx = 2^9
fhn = FitzHughNagumoModel(
    spatial_domain=Ω, time_domain=(0.0,4.0), Δx=(Ω[2] - 1/Nx)/Nx, Δt=dt,
    alpha_input_params=[500, 50000], beta_input_params=[10, 15]
)

# Some options for operator inference
options = LnL.LSOpInfOption(
    system=LnL.SystemStructure(
        state=[1, 2],      # linear and quadratic state terms
        control=1,         # linear input term B*u
        output=1,          # linear output term C*x
        coupled_input=1,   # bilinear term N*x*u
        constant=1,        # constant term K
    ),
    vars=LnL.VariableStructure(
        N=2,       # number of original variables (u, v)
        N_lift=3,  # number of lifted variables (u, v, w=u^2)
    ),
    data=LnL.DataStructure(
        Δt=dt,
        DS=100,
    ),
    optim=LnL.OptimizationSetting(
        verbose=true,
        nonredundant_operators=true,
        reproject=true,
    ),
)

# grid points
gp = fhn.spatial_dim

# Downsampling
DS = options.data.DS

##########################################################
## Get the full-order model operators for intrusive model
##########################################################
# Lifted (quadratic-bilinear) full-order operators: A, B, C, A2 (acting on w⊗w), N, K
tmp = fhn.lifted_finite_diff_model(gp, fhn.spatial_domain[2])
fomLinOps = LnL.Operators(
    A=tmp[1],
    B=tmp[2][:, :],  # Make sure its matrix
    C=tmp[3][:, :],
    A2=tmp[4],
    N=tmp[5],
    K=tmp[6]
)

@info "(t=$(time()-start)) Complete generating intrusive model operators"

# Generic function for FOM
tmp = fhn.full_order_model(gp, fhn.spatial_domain[2])  # much efficient to calculate for FOM
fomOps = LnL.Operators(
    A=tmp[1],
    B=tmp[2][:, :],  # Make sure its a matrix
    C=tmp[3][:, :],
    K=tmp[4],
    f=tmp[5]         # cubic nonlinearity f(x, u)
)
fom_state(x, u) = fomOps.A * x + fomOps.B * u + fomOps.f(x, u) + fomOps.K

@info "(t=$(time()-start)) Complete generating full-order model operators"

## Generate training data

The full-order model is integrated with the forward Euler scheme of `fhn.integrate_model` using the right-hand side `fom_state`. We store every `DS`-th snapshot (every 0.01 s).

In [ ]:
# parameters for the training data
α_train = vec([500 5000 50000] .* ones(3))
β_train = vec([10 12.5 15]' .* ones(3)')

###################### (I) Create training data ###########################
Xtrain = Vector{Matrix{Float64}}(undef, length(α_train))
Utrain = Vector{Matrix{Float64}}(undef, length(α_train))
Utrain_all = Vector{Matrix{Float64}}(undef, length(α_train))

@showprogress for i in axes(α_train, 1)
    α, β = α_train[i], β_train[i]
    genU(t) = α * t^3 * exp(-β * t)  # generic function for input

    ## training data for inferred dynamical models
    X = fhn.integrate_model(fhn.tspan, fhn.IC, genU; functional=fom_state)
    Xtrain[i] = X[:, 1:DS:end]  # make sure to only record every 0.01s
    U = genU.(fhn.tspan)
    Utrain_all[i] = U'
    Utrain[i] = U[1:DS:end]'
end
Xtr = reduce(hcat, Xtrain)
Utr = reshape(reduce(hcat, Utrain), :, 1)  # make sure that the U matrix is a tall matrix (x-row & t-col)
Ytr = fomOps.C * Xtr

## Check that the training data looks fine

In [ ]:
p = plot(legend=false, xlabel="t", ylabel="u(0,t), v(0,t)")
for i in eachindex(Xtrain)
    plot!(p, fhn.tspan[1:DS:end], Xtrain[i][1, :])     # u at the left boundary
    plot!(p, fhn.tspan[1:DS:end], Xtrain[i][gp+1, :])  # v at the left boundary
end
display(p)

In [ ]:
# Contour plot
p1 = contourf(fhn.tspan[1:DS:end],fhn.xspan,Xtrain[1][1:gp, :],lw=0, xlabel="t", ylabel="x", title="x1")
p2 = contourf(fhn.tspan[1:DS:end],fhn.xspan,Xtrain[1][gp+1:end, :],lw=0, xlabel="t", ylabel="x", title="x2")

lay = @layout [a b]
p = plot(p1, p2, layout=lay, size=(900, 400))
display(p)

## Generate test data

In [ ]:
################### (II) Create testing data 1 ############################
N_tests = 16

# Parameters for the testing data (1): random parameters inside the training range
α_test1 = vec(rand(N_tests, 1) .* (fhn.alpha_input_params[2] - fhn.alpha_input_params[1]) .+ fhn.alpha_input_params[1])
β_test1 = vec(rand(N_tests, 1) .* (fhn.beta_input_params[2] - fhn.beta_input_params[1]) .+ fhn.beta_input_params[1])

Xtest1 = Vector{Matrix{Float64}}(undef, N_tests)
Utest1 = Vector{Matrix{Float64}}(undef, N_tests)
Utest1_all = Vector{Matrix{Float64}}(undef, N_tests)
@showprogress for i in axes(α_test1, 1)
    α, β = α_test1[i], β_test1[i]
    genU(t) = α * t^3 * exp(-β * t)  # generic function for input

    X = fhn.integrate_model(fhn.tspan, fhn.IC, genU; functional=fom_state)
    Xtest1[i] = X[:, 1:DS:end]  # make sure to only record every 0.01s
    U = genU.(fhn.tspan)
    Utest1_all[i] = U'
    Utest1[i] = U[1:DS:end]'
end

In [ ]:
################### (III) Create testing data 2 ###########################
N_test_sqrt = Int(sqrt(N_tests))
# Parameters for testing data (2): a grid of parameters outside the training range
α_test2 = vec((5 * 10 .^ range(start=4, stop=6, length=N_test_sqrt))' .* ones(N_test_sqrt))
β_test2 = vec(range(start=15, stop=20, length=N_test_sqrt) .* ones(N_test_sqrt)')

Xtest2 = Vector{Matrix{Float64}}(undef, N_tests)
Utest2 = Vector{Matrix{Float64}}(undef, N_tests)
Utest2_all = Vector{Matrix{Float64}}(undef, N_tests)
@showprogress for i in axes(α_test2, 1)
    α, β = α_test2[i], β_test2[i]
    genU(t) = α * t^3 * exp(-β * t)  # generic function for input

    X = fhn.integrate_model(fhn.tspan, fhn.IC, genU; functional=fom_state)
    Xtest2[i] = X[:, 1:DS:end]  # make sure to only record every 0.01s
    U = genU.(fhn.tspan)
    Utest2_all[i] = U'
    Utest2[i] = U[1:DS:end]'
end

## Learn and Test

We lift the training data with the lifting map $w = u^2$ (`LnL.lifting`) and build a block-diagonal POD basis with one block per lifted variable.

The derivative data is generated by **reprojection** (`reproject=true` in the options): each reduced snapshot is reconstructed in the lifted space, the time derivative of the original variables $(u, v)$ is evaluated with the full-order right-hand side of `fomOps` (including the user-supplied cubic nonlinearity `fomOps.f`), the derivative of the lifted variable $w$ is obtained by a small forward-Euler step through the lifting map, and the result is projected back onto the basis. `LnL.opinf(Wtr, Vr, lifter, fomOps, options; U, Y)` does all of this and then infers the operators by least squares.

In [ ]:
##################### (IV~VII) Learn & Analysis ###########################
mode_req = [1 1 1; 2 1 3; 3 3 4; 5 4 5]  # Required number of modes for each lifted variables

# Data lifting
Xsep = [Xtr[1:gp, :], Xtr[gp+1:end, :]]
lifter = LnL.lifting(options.vars.N, options.vars.N_lift, [x -> x[1] .^ 2])
Wtr = lifter.map(Xsep)

# Take the SVD for each variable
W1 = svd(Wtr[1:gp, :])
W2 = svd(Wtr[gp+1:2*gp, :])
W3 = svd(Wtr[2*gp+1:end, :])

# dictionary with intrusive and LnL errors as matrices (9-by-4)
train_err = Dict(
    :intrusive => zeros(length(α_train), size(mode_req, 1)),
    :inferred => zeros(length(α_train), size(mode_req, 1))
)
test1_err = Dict(
    :intrusive => zeros(N_tests, size(mode_req, 1)),
    :inferred => zeros(N_tests, size(mode_req, 1))
)
test2_err = Dict(
    :intrusive => zeros(N_tests, size(mode_req, 1)),
    :inferred => zeros(N_tests, size(mode_req, 1))
)

@showprogress for (i, row) in enumerate(eachrow(mode_req))
    r1, r2, r3 = row
    Vr1 = W1.U[:, 1:r1]
    Vr2 = W2.U[:, 1:r2]
    Vr3 = W3.U[:, 1:r3]
    Vr = BlockDiagonal([Vr1, Vr2, Vr3])

    # Lift & Learn: OpInf with reprojected derivative data
    infOps = LnL.opinf(Wtr, Vr, lifter, fomOps, options; U=Utr, Y=Ytr)

    # Intrusive POD-Galerkin projection of the lifted full-order operators
    intruOps = LnL.pod(fomLinOps, Vr, options.system; nonredundant_operators=false)

    finf = (x, u) -> infOps.A * x + infOps.B * u + infOps.A2u * (x ⊘ x) + (infOps.N * x) * u[1] + infOps.K
    fint = (x, u) -> intruOps.A * x + intruOps.B * u + intruOps.A2u * (x ⊘ x) + (intruOps.N * x) * u[1] + intruOps.K

    k, l = 0, 0
    for (X, U) in zip(Xtrain, Utrain_all)
        Xint = fhn.integrate_model(fhn.tspan, Vr' * fhn.IC_lift, U; functional=fint)
        Xinf = fhn.integrate_model(fhn.tspan, Vr' * fhn.IC_lift, U; functional=finf)

        # Down sample
        Xint = Xint[:, 1:DS:end]
        Xinf = Xinf[:, 1:DS:end]

        train_err[:intrusive][k+=1, i] = LnL.rel_state_error(X, Xint, Vr)
        train_err[:inferred][l+=1, i] = LnL.rel_state_error(X, Xinf, Vr)
    end

    k, l = 0, 0
    for (X, U) in zip(Xtest1, Utest1_all)
        Xint = fhn.integrate_model(fhn.tspan, Vr' * fhn.IC_lift, U; functional=fint)
        Xinf = fhn.integrate_model(fhn.tspan, Vr' * fhn.IC_lift, U; functional=finf)

        # Down sample
        Xint = Xint[:, 1:DS:end]
        Xinf = Xinf[:, 1:DS:end]

        test1_err[:intrusive][k+=1, i] = LnL.rel_state_error(X, Xint, Vr)
        test1_err[:inferred][l+=1, i] = LnL.rel_state_error(X, Xinf, Vr)
    end

    k, l = 0, 0
    for (X, U) in zip(Xtest2, Utest2_all)
        Xint = fhn.integrate_model(fhn.tspan, Vr' * fhn.IC_lift, U; functional=fint)
        Xinf = fhn.integrate_model(fhn.tspan, Vr' * fhn.IC_lift, U; functional=finf)

        # Down sample
        Xint = Xint[:, 1:DS:end]
        Xinf = Xinf[:, 1:DS:end]

        test2_err[:intrusive][k+=1, i] = LnL.rel_state_error(X, Xint, Vr)
        test2_err[:inferred][l+=1, i] = LnL.rel_state_error(X, Xinf, Vr)
    end
end
dims = sum(mode_req, dims=2)

## Plot results

`LnL.rel_state_error` compares the original (unlifted) state $(u, v)$ of the full-order model with the reduced solution mapped back by the first $2 \cdot$`gp` rows of the basis. For test set 2 we take the median ignoring `NaN`s, in case a reduced model diverges for parameters far outside the training range.

In [ ]:
err_intru = vec(median(train_err[:intrusive], dims=1))
err_infer = vec(median(train_err[:inferred], dims=1))

t1err_intru = vec(median(test1_err[:intrusive], dims=1))
t1err_infer = vec(median(test1_err[:inferred], dims=1))

t2err_intru = vec(nanmedian(test2_err[:intrusive], dims=1))
t2err_infer = vec(nanmedian(test2_err[:inferred], dims=1));

In [ ]:
plot(dims, err_intru, marker=(:cross, 10), label="intru")
plot!(dims, err_infer, marker=(:circle), ls=:dash, label="opinf")
plot!(yscale=:log10, majorgrid=true, minorgrid=true)
tmp = log10.(err_infer)
yticks!([10.0^i for i in floor(minimum(tmp))-1:ceil(maximum(tmp))+1])
xticks!(vec(dims))
xlabel!("dimension n")
ylabel!("Relative State Error")
title!("Median Error over Training Trajectories")

In [ ]:
plot(dims, t1err_intru, marker=(:cross, 10), label="intru")
plot!(dims, t1err_infer, marker=(:circle), ls=:dash, label="opinf")
plot!(yscale=:log10, majorgrid=true, minorgrid=true)
tmp = log10.(t1err_infer)
yticks!([10.0^i for i in floor(minimum(tmp))-1:ceil(maximum(tmp))+1])
xticks!(vec(dims))
xlabel!("dimension n")
ylabel!("Relative State Error")
title!("Median Test1 Error over New Trajectories")

In [ ]:
plot(dims, t2err_intru, marker=(:cross, 10), label="intru")
plot!(dims, t2err_infer, marker=(:circle), ls=:dash, label="opinf")
plot!(yscale=:log10, majorgrid=true, minorgrid=true)
tmp = log10.(t2err_infer)
yticks!([10.0^i for i in floor(minimum(tmp))-1:ceil(maximum(tmp))+1])
xticks!(vec(dims))
xlabel!("dimension n")
ylabel!("Relative State Error")
title!("Median Test2 Error over New Trajectories")

## Discussion

- The Lift & Learn models (`opinf`) reproduce the accuracy of the intrusive POD-Galerkin models of the lifted system (`intru`) on the training trajectories and on test set 1 (parameters inside the training range), without accessing the lifted full-order operators.
- The errors generally decrease as the reduced dimension grows. The bump at $n = 6$ is caused by the chosen per-variable mode split (2 modes for $u$, 1 for $v$, 3 for $w$) and appears for the intrusive model as well.
- Test set 2 extrapolates in the input parameters, so the errors are larger. A few of the inferred models can become unstable for some of these trajectories; their `NaN` errors are ignored by `nanmedian`.